<a href="https://colab.research.google.com/github/samfozzy/AI/blob/main/circular_economy_simulation_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ♻️ Circular Economy Bootcamp — Simulation Toolkit
### *Plastics Recycling & Waste-to-Feed Optimization*

---

This notebook contains **two interactive simulation modules**:

| Module | Topic |
|--------|-------|
| **Module 1** | 🤖 AI-Assisted Plastic Sorting Simulation |
| **Module 2** | 🐄 Waste-to-Feed Optimization |

> **How to use:** Adjust parameters in the configuration cells, then run all cells (`Runtime → Run All`).

###### By Dr. SAMIUEL C. ATUAHENE
---

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SETUP — Run this first to install any missing libraries
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

try:
    import ipywidgets
except ImportError:
    install('ipywidgets')

import warnings
warnings.filterwarnings('ignore')
print('✅ Setup complete — ready to simulate!')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CORE IMPORTS & GLOBAL STYLE
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

plt.rcParams.update({
    'figure.facecolor':  '#0f1117',
    'axes.facecolor':    '#1a1d27',
    'axes.edgecolor':    '#2e3250',
    'axes.labelcolor':   '#c8d0e7',
    'xtick.color':       '#8892b0',
    'ytick.color':       '#8892b0',
    'text.color':        '#e0e6f0',
    'grid.color':        '#2e3250',
    'grid.linewidth':    0.6,
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    13,
    'axes.labelsize':    11,
})

C = {
    'teal':   '#00d4aa',
    'blue':   '#4f8ef7',
    'amber':  '#f5a623',
    'red':    '#ff6b6b',
    'green':  '#52e0a0',
    'purple': '#9b59b6',
    'grey':   '#8892b0',
    'bg':     '#0f1117',
    'card':   '#1a1d27',
}

print('✅ Libraries loaded.')

---
## 🤖 Module 1 — AI-Assisted Plastic Sorting Simulation

Simulates a **recycling facility processing 1,000 kg/day** of mixed plastic waste.  
Compare the economic and environmental outcomes of **manual sorting** vs **AI-assisted sorting**.

---

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 1 — USER-ADJUSTABLE PARAMETERS  ← Tweak these!
# ═════════════════════════════════════════════════════════════════════════════

# --- Facility ---
DAILY_INPUT_KG          = 1_000   # Total plastic waste processed per day (kg)
OPERATING_DAYS_PER_YEAR = 250     # Working days per year

# --- Sorting accuracy ---
MANUAL_ACCURACY         = 0.85    # Fraction correctly sorted (manual)
AI_ACCURACY             = 0.96    # Fraction correctly sorted (AI)

# --- Market prices (USD / kg) ---
PRICE_CLEAN_PLASTIC     = 0.60    # Revenue per kg of clean (correctly sorted) plastic
PRICE_CONTAMINATED      = 0.15    # Revenue per kg of contaminated plastic
LANDFILL_COST           = 0.08    # Cost per kg sent to landfill

# --- Environmental ---
CO2_SAVING_PER_KG       = 1.5     # kg CO2 saved per kg plastic recycled

# --- AI system economics ---
AI_DAILY_OPERATING_COST = 150     # USD/day for AI system (electricity, maintenance)
AI_CAPITAL_COST         = 120_000 # One-time capital investment for AI system
AI_SYSTEM_LIFETIME_DAYS = 365 * 5 # Expected system lifetime (5 years)

print(f"""
📋 Module 1 — Simulation Parameters
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Daily input          : {DAILY_INPUT_KG:,} kg
  Manual accuracy      : {MANUAL_ACCURACY*100:.0f}%
  AI accuracy          : {AI_ACCURACY*100:.0f}%
  Clean plastic price  : ${PRICE_CLEAN_PLASTIC:.2f}/kg
  Contaminated price   : ${PRICE_CONTAMINATED:.2f}/kg
  Landfill cost        : ${LANDFILL_COST:.2f}/kg
  CO₂ saving factor    : {CO2_SAVING_PER_KG} kg CO₂/kg recycled
  AI capital cost      : ${AI_CAPITAL_COST:,}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MODULE 1 — SIMULATION ENGINE
# ─────────────────────────────────────────────────────────────────────────────

def simulate_sorting(accuracy, daily_input, price_clean, price_contaminated,
                     landfill_cost, co2_per_kg, operating_days):
    """Simulate one sorting scenario. Returns dict of daily & annual metrics."""
    clean_kg          = daily_input * accuracy
    contaminated_kg   = daily_input * (1 - accuracy)
    contaminated_sold = contaminated_kg * 0.50   # 50% recoverable at low value
    landfill_kg       = contaminated_kg * 0.50   # 50% goes to landfill

    daily_revenue  = (clean_kg * price_clean) + (contaminated_sold * price_contaminated)
    daily_landfill = landfill_kg * landfill_cost
    daily_net      = daily_revenue - daily_landfill

    daily_co2_saved    = clean_kg * co2_per_kg
    daily_co2_landfill = landfill_kg * 0.9        # ~0.9 kg CO₂e per kg landfilled

    return {
        'accuracy':            accuracy,
        'daily_input':         daily_input,
        'clean_kg':            clean_kg,
        'contaminated_kg':     contaminated_kg,
        'contaminated_sold':   contaminated_sold,
        'landfill_kg':         landfill_kg,
        'daily_revenue':       daily_revenue,
        'daily_landfill_cost': daily_landfill,
        'daily_net':           daily_net,
        'annual_net':          daily_net * operating_days,
        'daily_co2_saved':     daily_co2_saved,
        'daily_co2_landfill':  daily_co2_landfill,
        'annual_co2_net':      (daily_co2_saved - daily_co2_landfill) * operating_days,
    }


def compute_ai_economics(manual, ai, ai_daily_opex, ai_capex, ai_lifetime_days, op_days):
    """Compare AI vs manual scenario economics."""
    daily_revenue_gain = ai['daily_net'] - manual['daily_net']
    ai_daily_capex     = ai_capex / ai_lifetime_days
    daily_net_gain     = daily_revenue_gain - ai_daily_opex - ai_daily_capex
    annual_net_gain    = daily_net_gain * op_days
    payback_days       = ai_capex / max(daily_net_gain, 0.01)
    annual_co2_diff    = ai['annual_co2_net'] - manual['annual_co2_net']
    contamination_reduction = (
        (manual['contaminated_kg'] - ai['contaminated_kg']) / manual['contaminated_kg'] * 100
    )
    return {
        'daily_revenue_gain':      daily_revenue_gain,
        'daily_net_gain':          daily_net_gain,
        'annual_net_gain':         annual_net_gain,
        'payback_days':            payback_days,
        'annual_co2_diff':         annual_co2_diff,
        'contamination_reduction': contamination_reduction,
    }


# Run baseline simulations
manual_result = simulate_sorting(
    MANUAL_ACCURACY, DAILY_INPUT_KG, PRICE_CLEAN_PLASTIC,
    PRICE_CONTAMINATED, LANDFILL_COST, CO2_SAVING_PER_KG, OPERATING_DAYS_PER_YEAR
)
ai_result = simulate_sorting(
    AI_ACCURACY, DAILY_INPUT_KG, PRICE_CLEAN_PLASTIC,
    PRICE_CONTAMINATED, LANDFILL_COST, CO2_SAVING_PER_KG, OPERATING_DAYS_PER_YEAR
)
econ = compute_ai_economics(
    manual_result, ai_result,
    AI_DAILY_OPERATING_COST, AI_CAPITAL_COST,
    AI_SYSTEM_LIFETIME_DAYS, OPERATING_DAYS_PER_YEAR
)

print('✅ Module 1 simulation engine ready.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MODULE 1 — PRINTED COMPARISON REPORT
# ─────────────────────────────────────────────────────────────────────────────

def print_module1_report(manual, ai, econ, op_days=OPERATING_DAYS_PER_YEAR):
    sep  = '═' * 60
    sep2 = '─' * 60
    print(f"""
{sep}
 ♻️  MODULE 1 — PLASTIC SORTING SIMULATION REPORT
{sep}

  📦 DAILY MATERIAL FLOWS (per {manual['daily_input']:,} kg input)
{sep2}
                           Manual      AI-Assisted
  Accuracy                 {manual['accuracy']*100:.0f}%          {ai['accuracy']*100:.0f}%
  Clean plastic (kg)       {manual['clean_kg']:>7,.1f}      {ai['clean_kg']:>7,.1f}
  Contaminated (kg)        {manual['contaminated_kg']:>7,.1f}      {ai['contaminated_kg']:>7,.1f}
  Sent to landfill (kg)    {manual['landfill_kg']:>7,.1f}      {ai['landfill_kg']:>7,.1f}

  💰 DAILY FINANCIALS
{sep2}
  Gross revenue            ${manual['daily_revenue']:>7,.2f}      ${ai['daily_revenue']:>7,.2f}
  Landfill cost            ${manual['daily_landfill_cost']:>7,.2f}      ${ai['daily_landfill_cost']:>7,.2f}
  Net revenue              ${manual['daily_net']:>7,.2f}      ${ai['daily_net']:>7,.2f}

  📅 ANNUAL FINANCIALS (over {op_days} operating days)
{sep2}
  Annual net revenue       ${manual['annual_net']:>10,.0f}   ${ai['annual_net']:>10,.0f}
  Annual revenue gain (AI)              ${econ['annual_net_gain']:>10,.0f}
  AI payback period        {econ['payback_days']:.0f} days  (~{econ['payback_days']/op_days:.1f} operating years)

  🌍 ENVIRONMENTAL IMPACT
{sep2}
  Daily CO₂ saved (kg)     {manual['daily_co2_saved']:>7,.1f}      {ai['daily_co2_saved']:>7,.1f}
  Annual CO₂ net (tonnes)  {manual['annual_co2_net']/1000:>7,.1f}      {ai['annual_co2_net']/1000:>7,.1f}
  Additional CO₂ avoided               +{econ['annual_co2_diff']/1000:,.1f} t CO₂/yr
  Contamination reduction               {econ['contamination_reduction']:.1f}%

{sep}
  🏆 KEY RESULTS
  → AI sorting earns  ${econ['annual_net_gain']:,.0f} MORE per year (net of AI costs)
  → Contamination reduced by {econ['contamination_reduction']:.1f}%
  → Additional {econ['annual_co2_diff']/1000:,.1f} tonnes CO₂ avoided annually
  → Capital investment recoups in ~{econ['payback_days']:.0f} days
{sep}
""")

print_module1_report(manual_result, ai_result, econ)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MODULE 1 — VISUALISATION DASHBOARD
# ─────────────────────────────────────────────────────────────────────────────

def plot_module1(manual, ai, econ, op_days=OPERATING_DAYS_PER_YEAR,
                 ai_capex=AI_CAPITAL_COST, ai_daily_opex=AI_DAILY_OPERATING_COST):
    fig = plt.figure(figsize=(16, 10), facecolor=C['bg'])
    fig.suptitle('♻️  Module 1 — AI vs Manual Plastic Sorting',
                 fontsize=18, color='white', fontweight='bold', y=0.98)
    gs = GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.38)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 2])
    ax4 = fig.add_subplot(gs[1, 0])
    ax5 = fig.add_subplot(gs[1, 1])
    ax6 = fig.add_subplot(gs[1, 2])

    labels = ['Manual', 'AI-Assisted']
    x  = np.arange(2)
    bw = 0.45
    scenarios = [manual, ai]

    # 1 ── Material flow grouped bar
    ax1.set_title('Daily Material Flow (kg)', color='white')
    bar_w = 0.25
    for i, (key, col, lbl) in enumerate([
        ('clean_kg',          C['teal'],  'Clean'),
        ('contaminated_sold', C['amber'], 'Contaminated'),
        ('landfill_kg',       C['red'],   'Landfill'),
    ]):
        vals = [s[key] for s in scenarios]
        bars = ax1.bar(x + i*bar_w - bar_w, vals, bar_w, label=lbl, color=col, alpha=0.85)
        for b, v in zip(bars, vals):
            ax1.text(b.get_x()+b.get_width()/2, b.get_height()+5,
                     f'{v:.0f}', ha='center', fontsize=7.5, color='white')
    ax1.set_xticks(x); ax1.set_xticklabels(labels)
    ax1.legend(fontsize=8, framealpha=0.2); ax1.set_ylabel('kg / day'); ax1.grid(axis='y')

    # 2 ── Daily net revenue
    ax2.set_title('Daily Net Revenue (USD)', color='white')
    vals = [s['daily_net'] for s in scenarios]
    bars = ax2.bar(labels, vals, color=[C['blue'], C['teal']], width=bw, alpha=0.9)
    for b, v in zip(bars, vals):
        ax2.text(b.get_x()+b.get_width()/2, b.get_height()+1,
                 f'${v:,.0f}', ha='center', fontsize=10, color='white', fontweight='bold')
    ax2.set_ylabel('USD / day'); ax2.grid(axis='y')

    # 3 ── CO2 avoided per day
    ax3.set_title('Net CO₂ Avoided / Day (kg)', color='white')
    vals = [s['daily_co2_saved'] - s['daily_co2_landfill'] for s in scenarios]
    bars = ax3.bar(labels, vals, color=[C['purple'], C['green']], width=bw, alpha=0.9)
    for b, v in zip(bars, vals):
        ax3.text(b.get_x()+b.get_width()/2, b.get_height()+2,
                 f'{v:,.0f} kg', ha='center', fontsize=9.5, color='white', fontweight='bold')
    ax3.set_ylabel('kg CO₂ / day'); ax3.grid(axis='y')

    # 4 ── Stacked composition (%)
    ax4.set_title('Plastic Fate (% of daily input)', color='white')
    total = manual['daily_input']
    clean_pct  = [s['clean_kg']/total*100          for s in scenarios]
    cont_pct   = [s['contaminated_sold']/total*100  for s in scenarios]
    lf_pct     = [s['landfill_kg']/total*100        for s in scenarios]
    ax4.bar(labels, clean_pct, color=C['teal'],  alpha=0.9, label='Clean recycled')
    ax4.bar(labels, cont_pct,  color=C['amber'], alpha=0.9, label='Contaminated sold',
            bottom=clean_pct)
    ax4.bar(labels, lf_pct,    color=C['red'],   alpha=0.9, label='Landfill',
            bottom=[c+ct for c, ct in zip(clean_pct, cont_pct)])
    ax4.set_ylim(0, 108); ax4.set_ylabel('% of input')
    ax4.legend(fontsize=8, framealpha=0.2); ax4.grid(axis='y')

    # 5 ── Annual net revenue
    ax5.set_title('Annual Net Revenue (USD)', color='white')
    vals = [s['annual_net'] for s in scenarios]
    bars = ax5.bar(labels, vals, color=[C['blue'], C['teal']], width=bw, alpha=0.9)
    for b, v in zip(bars, vals):
        ax5.text(b.get_x()+b.get_width()/2, b.get_height()+200,
                 f'${v:,.0f}', ha='center', fontsize=9, color='white', fontweight='bold')
    mid_y = (vals[0] + vals[1]) / 2
    ax5.annotate(f'Δ +${econ["annual_net_gain"]:,.0f}/yr\n(net of AI costs)',
                 xy=(1, vals[1]), xytext=(0.5, mid_y),
                 color=C['teal'], fontsize=8.5, ha='center',
                 arrowprops=dict(arrowstyle='->', color=C['teal']))
    ax5.set_ylabel('USD / year'); ax5.grid(axis='y')

    # 6 ── Payback curve
    ax6.set_title('AI Investment Payback Curve', color='white')
    max_days = min(int(econ['payback_days'] * 2) + 1, 1500)
    days = np.arange(0, max_days)
    cumulative = econ['daily_net_gain'] * days
    ax6.plot(days, cumulative, color=C['teal'], linewidth=2.5, label='Cumulative net gain')
    ax6.axhline(0,          color=C['grey'],  linewidth=1,   linestyle='--')
    ax6.axhline(ai_capex,   color=C['amber'], linewidth=1.5, linestyle='--',
                label=f'Capital cost (${ai_capex:,})')
    if econ['payback_days'] < max_days:
        ax6.axvline(econ['payback_days'], color=C['red'], linewidth=1.5,
                    linestyle=':', label=f'Payback ≈ {econ["payback_days"]:.0f} days')
        ax6.scatter([econ['payback_days']], [ai_capex], color=C['red'], s=80, zorder=5)
    ax6.set_xlabel('Days since AI deployment')
    ax6.set_ylabel('Cumulative USD')
    ax6.legend(fontsize=8, framealpha=0.2); ax6.grid()

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


plot_module1(manual_result, ai_result, econ)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MODULE 1 — INTERACTIVE WIDGET CONTROLS
# ─────────────────────────────────────────────────────────────────────────────

style  = {'description_width': '230px'}
layout = widgets.Layout(width='560px')

w_input  = widgets.IntSlider(value=1000,  min=100,  max=5000, step=100,
               description='Daily input (kg)',           style=style, layout=layout)
w_manual = widgets.FloatSlider(value=0.85, min=0.50, max=0.99, step=0.01,
               description='Manual accuracy',            style=style, layout=layout,
               readout_format='.0%')
w_ai     = widgets.FloatSlider(value=0.96, min=0.80, max=1.00, step=0.01,
               description='AI accuracy',                style=style, layout=layout,
               readout_format='.0%')
w_clean  = widgets.FloatSlider(value=0.60, min=0.10, max=2.00, step=0.05,
               description='Clean plastic price ($/kg)', style=style, layout=layout)
w_contam = widgets.FloatSlider(value=0.15, min=0.01, max=0.50, step=0.01,
               description='Contaminated price ($/kg)',  style=style, layout=layout)
w_lf     = widgets.FloatSlider(value=0.08, min=0.01, max=0.30, step=0.01,
               description='Landfill cost ($/kg)',       style=style, layout=layout)

out1 = widgets.Output()

def update_m1(**kw):
    with out1:
        clear_output(wait=True)
        m = simulate_sorting(kw['manual'], kw['inp'], kw['clean'], kw['contam'],
                             kw['lf'], CO2_SAVING_PER_KG, OPERATING_DAYS_PER_YEAR)
        a = simulate_sorting(kw['ai'],     kw['inp'], kw['clean'], kw['contam'],
                             kw['lf'], CO2_SAVING_PER_KG, OPERATING_DAYS_PER_YEAR)
        e = compute_ai_economics(m, a, AI_DAILY_OPERATING_COST, AI_CAPITAL_COST,
                                 AI_SYSTEM_LIFETIME_DAYS, OPERATING_DAYS_PER_YEAR)
        print_module1_report(m, a, e)
        plot_module1(m, a, e)

ip1 = widgets.interactive(update_m1,
    inp    = w_input,
    manual = w_manual,
    ai     = w_ai,
    clean  = w_clean,
    contam = w_contam,
    lf     = w_lf,
)

display(HTML('<h3 style="color:#00d4aa; font-family:monospace">🎛️ Interactive Controls — Module 1</h3>'))
display(ip1)
display(out1)

---
## 🐄 Module 2 — Waste-to-Feed Optimization

Optimises how **organic food waste** is diverted from landfill into **animal feed, biogas, or compost**, maximising value recovery and minimising environmental impact.

---

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  MODULE 2 — USER-ADJUSTABLE PARAMETERS  ← Tweak these!
# ═════════════════════════════════════════════════════════════════════════════

# --- Facility ---
M2_DAILY_WASTE_KG    = 2_000   # Organic food waste input per day (kg)
M2_OPERATING_DAYS    = 300     # Operating days per year

# --- Optimised waste routing (fractions must sum to 1.0) ---
M2_FRAC_FEED         = 0.45   # High-quality → animal feed
M2_FRAC_BIOGAS       = 0.30   # Wet/mixed → anaerobic digestion
M2_FRAC_COMPOST      = 0.20   # Dry/fibrous → compost
M2_FRAC_LANDFILL     = 0.05   # Unavoidable residual

assert abs(M2_FRAC_FEED + M2_FRAC_BIOGAS + M2_FRAC_COMPOST + M2_FRAC_LANDFILL - 1.0) < 0.001, \
    "⚠️  Fractions must sum to 1.0!"

# --- Revenue & costs (USD / kg) ---
M2_PRICE_FEED        = 0.35   # Processed animal feed
M2_PRICE_BIOGAS      = 0.20   # Equivalent biogas value
M2_PRICE_COMPOST     = 0.08   # Compost sold
M2_LANDFILL_COST     = 0.12   # Landfill disposal cost
M2_PROC_FEED         = 0.10   # Processing cost — feed
M2_PROC_BIO          = 0.07   # Processing cost — biogas
M2_PROC_COMP         = 0.04   # Processing cost — compost

# --- Environmental factors (kg CO₂e per kg waste) ---
M2_CO2_LANDFILL      = 0.45   # Methane from organic landfill waste
M2_CO2_FEED          = 0.80   # Avoided feed-crop emissions
M2_CO2_BIOGAS        = 0.60   # Replaces fossil gas
M2_CO2_COMPOST       = 0.25   # Carbon sequestration & soil benefit

# --- Baseline scenario (pre-optimisation) ---
M2_BASELINE_LF_FRAC  = 0.80   # 80% landfill is common in developing markets

print(f"""
📋 Module 2 — Simulation Parameters
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Daily waste input    : {M2_DAILY_WASTE_KG:,} kg
  Optimised routing:
    → Feed             : {M2_FRAC_FEED*100:.0f}%
    → Biogas           : {M2_FRAC_BIOGAS*100:.0f}%
    → Compost          : {M2_FRAC_COMPOST*100:.0f}%
    → Landfill         : {M2_FRAC_LANDFILL*100:.0f}%
  Prices: feed ${M2_PRICE_FEED}/kg | biogas ${M2_PRICE_BIOGAS}/kg | compost ${M2_PRICE_COMPOST}/kg
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MODULE 2 — SIMULATION ENGINE
# ─────────────────────────────────────────────────────────────────────────────

def simulate_waste_to_feed(daily_waste, frac_feed, frac_biogas, frac_compost, frac_landfill,
                            price_feed, price_biogas, price_compost, lf_cost,
                            proc_feed, proc_bio, proc_comp,
                            co2_lf, co2_feed, co2_biogas, co2_compost, op_days, label=''):
    feed_kg    = daily_waste * frac_feed
    biogas_kg  = daily_waste * frac_biogas
    compost_kg = daily_waste * frac_compost
    lf_kg      = daily_waste * frac_landfill

    rev_feed    = feed_kg    * (price_feed    - proc_feed)
    rev_biogas  = biogas_kg  * (price_biogas  - proc_bio)
    rev_compost = compost_kg * (price_compost - proc_comp)
    cost_lf     = lf_kg * lf_cost
    daily_net   = rev_feed + rev_biogas + rev_compost - cost_lf

    co2_avoided   = feed_kg*co2_feed + biogas_kg*co2_biogas + compost_kg*co2_compost
    co2_emitted   = lf_kg * co2_lf
    daily_co2_net = co2_avoided - co2_emitted

    return {
        'label':          label,
        'daily_waste':    daily_waste,
        'feed_kg':        feed_kg,
        'biogas_kg':      biogas_kg,
        'compost_kg':     compost_kg,
        'lf_kg':          lf_kg,
        'rev_feed':       rev_feed,
        'rev_biogas':     rev_biogas,
        'rev_compost':    rev_compost,
        'cost_lf':        cost_lf,
        'daily_net':      daily_net,
        'annual_net':     daily_net * op_days,
        'co2_avoided':    co2_avoided,
        'co2_emitted':    co2_emitted,
        'daily_co2_net':  daily_co2_net,
        'annual_co2_net': daily_co2_net * op_days,
        'diversion_rate': (1 - frac_landfill) * 100,
    }


def _m2_kwargs(daily_waste, frac_feed, frac_biogas, frac_compost, frac_landfill, label):
    return dict(
        daily_waste=daily_waste, frac_feed=frac_feed, frac_biogas=frac_biogas,
        frac_compost=frac_compost, frac_landfill=frac_landfill,
        price_feed=M2_PRICE_FEED, price_biogas=M2_PRICE_BIOGAS, price_compost=M2_PRICE_COMPOST,
        lf_cost=M2_LANDFILL_COST, proc_feed=M2_PROC_FEED, proc_bio=M2_PROC_BIO,
        proc_comp=M2_PROC_COMP, co2_lf=M2_CO2_LANDFILL, co2_feed=M2_CO2_FEED,
        co2_biogas=M2_CO2_BIOGAS, co2_compost=M2_CO2_COMPOST,
        op_days=M2_OPERATING_DAYS, label=label
    )


m2_baseline  = simulate_waste_to_feed(**_m2_kwargs(
    M2_DAILY_WASTE_KG, 0.10, 0.05, 0.05, M2_BASELINE_LF_FRAC, 'Baseline'))
m2_optimised = simulate_waste_to_feed(**_m2_kwargs(
    M2_DAILY_WASTE_KG, M2_FRAC_FEED, M2_FRAC_BIOGAS, M2_FRAC_COMPOST, M2_FRAC_LANDFILL, 'Optimised'))

print('✅ Module 2 simulation engine ready.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MODULE 2 — PRINTED COMPARISON REPORT
# ─────────────────────────────────────────────────────────────────────────────

def print_module2_report(baseline, opt):
    sep  = '═' * 60
    sep2 = '─' * 60
    gain_annual = opt['annual_net']     - baseline['annual_net']
    co2_gain    = opt['annual_co2_net'] - baseline['annual_co2_net']
    div_gain    = opt['diversion_rate'] - baseline['diversion_rate']
    print(f"""
{sep}
 🐄  MODULE 2 — WASTE-TO-FEED OPTIMISATION REPORT
{sep}

  📦 DAILY MATERIAL ROUTING (per {baseline['daily_waste']:,} kg input)
{sep2}
                           Baseline    Optimised
  → Animal feed (kg)       {baseline['feed_kg']:>7,.0f}      {opt['feed_kg']:>7,.0f}
  → Biogas (kg)            {baseline['biogas_kg']:>7,.0f}      {opt['biogas_kg']:>7,.0f}
  → Compost (kg)           {baseline['compost_kg']:>7,.0f}      {opt['compost_kg']:>7,.0f}
  → Landfill (kg)          {baseline['lf_kg']:>7,.0f}      {opt['lf_kg']:>7,.0f}
  Diversion rate           {baseline['diversion_rate']:>6.0f}%      {opt['diversion_rate']:>6.0f}%

  💰 DAILY FINANCIALS
{sep2}
  Feed revenue             ${baseline['rev_feed']:>7,.2f}      ${opt['rev_feed']:>7,.2f}
  Biogas revenue           ${baseline['rev_biogas']:>7,.2f}      ${opt['rev_biogas']:>7,.2f}
  Compost revenue          ${baseline['rev_compost']:>7,.2f}      ${opt['rev_compost']:>7,.2f}
  Landfill cost           -${baseline['cost_lf']:>7,.2f}     -${opt['cost_lf']:>7,.2f}
  Daily net                ${baseline['daily_net']:>7,.2f}      ${opt['daily_net']:>7,.2f}

  📅 ANNUAL FINANCIALS (over {M2_OPERATING_DAYS} days)
{sep2}
  Annual net               ${baseline['annual_net']:>10,.0f}   ${opt['annual_net']:>10,.0f}
  Annual gain (optimised)               ${gain_annual:>10,.0f}

  🌍 ANNUAL ENVIRONMENTAL IMPACT
{sep2}
  Annual CO₂ net (tonnes)  {baseline['annual_co2_net']/1000:>7,.1f}      {opt['annual_co2_net']/1000:>7,.1f}
  Additional CO₂ avoided                +{co2_gain/1000:,.1f} tonnes/yr

{sep}
  🏆 KEY RESULTS
  → Landfill diversion improved by {div_gain:.0f} percentage points
  → Annual revenue gain: ${gain_annual:,.0f}
  → Additional CO₂ avoided: {co2_gain/1000:,.1f} tonnes/yr
  → Equivalent to removing ~{co2_gain/1000/2.3:.0f} cars from the road
{sep}
""")

print_module2_report(m2_baseline, m2_optimised)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MODULE 2 — VISUALISATION DASHBOARD
# ─────────────────────────────────────────────────────────────────────────────

def plot_module2(baseline, opt):
    fig = plt.figure(figsize=(16, 10), facecolor=C['bg'])
    fig.suptitle('🐄  Module 2 — Waste-to-Feed Optimisation Dashboard',
                 fontsize=18, color='white', fontweight='bold', y=0.98)
    gs = GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.38)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 2])
    ax4 = fig.add_subplot(gs[1, 0])
    ax5 = fig.add_subplot(gs[1, 1])
    ax6 = fig.add_subplot(gs[1, 2])

    labels    = ['Baseline', 'Optimised']
    scenarios = [baseline, opt]
    bw        = 0.45
    x         = np.arange(2)

    # 1 ── Material routing grouped bar
    ax1.set_title('Daily Material Routing (kg)', color='white')
    bar_w = 0.20
    for i, (key, col, lbl) in enumerate([
        ('feed_kg',    C['teal'],   'Feed'),
        ('biogas_kg',  C['purple'], 'Biogas'),
        ('compost_kg', C['green'],  'Compost'),
        ('lf_kg',      C['red'],    'Landfill'),
    ]):
        vals = [s[key] for s in scenarios]
        bars = ax1.bar(x + i*bar_w - 1.5*bar_w, vals, bar_w,
                       label=lbl, color=col, alpha=0.85)
        for b, v in zip(bars, vals):
            ax1.text(b.get_x()+b.get_width()/2, b.get_height()+5,
                     f'{v:.0f}', ha='center', fontsize=7, color='white')
    ax1.set_xticks(x); ax1.set_xticklabels(labels)
    ax1.legend(fontsize=8, framealpha=0.2, ncol=2)
    ax1.set_ylabel('kg / day'); ax1.grid(axis='y')

    # 2 ── Daily net revenue
    ax2.set_title('Daily Net Revenue (USD)', color='white')
    vals = [s['daily_net'] for s in scenarios]
    bars = ax2.bar(labels, vals, color=[C['blue'], C['teal']], width=bw, alpha=0.9)
    for b, v in zip(bars, vals):
        ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
                 f'${v:,.0f}', ha='center', fontsize=10, color='white', fontweight='bold')
    ax2.set_ylabel('USD / day'); ax2.grid(axis='y')

    # 3 ── CO2 net avoided
    ax3.set_title('Net CO₂ Avoided / Day (kg)', color='white')
    vals = [s['daily_co2_net'] for s in scenarios]
    bars = ax3.bar(labels, vals, color=[C['amber'], C['green']], width=bw, alpha=0.9)
    for b, v in zip(bars, vals):
        ax3.text(b.get_x()+b.get_width()/2, max(v+2, 2),
                 f'{v:,.0f} kg', ha='center', fontsize=9.5, color='white', fontweight='bold')
    ax3.set_ylabel('kg CO₂ avoided / day'); ax3.grid(axis='y')

    # 4 ── Pie chart (optimised routing)
    ax4.set_title('Optimised Waste Routing (% of input)', color='white')
    sizes  = [opt['feed_kg'], opt['biogas_kg'], opt['compost_kg'], opt['lf_kg']]
    clrs   = [C['teal'], C['purple'], C['green'], C['red']]
    lbls   = ['Feed', 'Biogas', 'Compost', 'Landfill']
    wedges, texts, autotexts = ax4.pie(
        sizes, labels=lbls, colors=clrs, autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor': C['bg'], 'linewidth': 2})
    for t in autotexts: t.set_color('white'); t.set_fontsize(9)
    for t in texts:     t.set_color(C['grey'])

    # 5 ── Stacked revenue breakdown
    ax5.set_title('Revenue Breakdown by Stream (USD/day)', color='white')
    bottoms = [0, 0]
    for key, col, lbl in [('rev_feed',C['teal'],'Feed'), ('rev_biogas',C['purple'],'Biogas'),
                           ('rev_compost',C['green'],'Compost')]:
        vals = [s[key] for s in scenarios]
        ax5.bar(labels, vals, bottom=bottoms, color=col, alpha=0.88, label=lbl)
        bottoms = [b+v for b, v in zip(bottoms, vals)]
    ax5.bar(labels, [-s['cost_lf'] for s in scenarios], color=C['red'],
            alpha=0.88, label='Landfill cost')
    ax5.axhline(0, color=C['grey'], linewidth=1)
    ax5.legend(fontsize=8, framealpha=0.2); ax5.set_ylabel('USD / day'); ax5.grid(axis='y')

    # 6 ── Cumulative CO2 avoided over year
    ax6.set_title('Cumulative CO₂ Avoided (tonnes/year)', color='white')
    days = np.arange(0, M2_OPERATING_DAYS + 1)
    b_line = baseline['daily_co2_net'] * days / 1000
    o_line = opt['daily_co2_net']      * days / 1000
    ax6.fill_between(days, b_line, alpha=0.25, color=C['blue'])
    ax6.fill_between(days, o_line, alpha=0.25, color=C['teal'])
    ax6.plot(days, b_line, color=C['blue'], linewidth=2, label='Baseline')
    ax6.plot(days, o_line, color=C['teal'], linewidth=2, label='Optimised')
    ax6.set_xlabel('Day of year'); ax6.set_ylabel('Tonnes CO₂ avoided')
    ax6.legend(fontsize=9, framealpha=0.2); ax6.grid()

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


plot_module2(m2_baseline, m2_optimised)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MODULE 2 — INTERACTIVE WIDGET CONTROLS
# ─────────────────────────────────────────────────────────────────────────────

style2  = {'description_width': '230px'}
layout2 = widgets.Layout(width='560px')

w2_waste   = widgets.IntSlider(  value=2000, min=500, max=10000, step=100,
                 description='Daily waste input (kg)', style=style2, layout=layout2)
w2_feed    = widgets.FloatSlider(value=0.45, min=0.0, max=0.90, step=0.01,
                 description='Feed fraction',          style=style2, layout=layout2,
                 readout_format='.0%')
w2_biogas  = widgets.FloatSlider(value=0.30, min=0.0, max=0.70, step=0.01,
                 description='Biogas fraction',        style=style2, layout=layout2,
                 readout_format='.0%')
w2_compost = widgets.FloatSlider(value=0.20, min=0.0, max=0.60, step=0.01,
                 description='Compost fraction',       style=style2, layout=layout2,
                 readout_format='.0%')

out2 = widgets.Output()

def update_m2(**kw):
    total   = kw['feed'] + kw['biogas'] + kw['compost']
    lf_frac = max(0.0, round(1.0 - total, 6))
    with out2:
        clear_output(wait=True)
        if total > 1.001:
            print(f'⚠️  Fractions sum to {total:.2f} — must be ≤ 1.0. Reduce a slider.')
            return
        print(f'ℹ️  Landfill fraction auto-set to {lf_frac*100:.1f}%')
        bsl = simulate_waste_to_feed(**_m2_kwargs(kw['waste'], 0.10, 0.05, 0.05,
                                                   M2_BASELINE_LF_FRAC, 'Baseline'))
        opt = simulate_waste_to_feed(**_m2_kwargs(kw['waste'], kw['feed'], kw['biogas'],
                                                   kw['compost'], lf_frac, 'Optimised'))
        print_module2_report(bsl, opt)
        plot_module2(bsl, opt)

ip2 = widgets.interactive(update_m2,
    waste   = w2_waste,
    feed    = w2_feed,
    biogas  = w2_biogas,
    compost = w2_compost,
)

display(HTML('<h3 style="color:#00d4aa; font-family:monospace">🎛️ Interactive Controls — Module 2</h3>'))
display(HTML('<p style="color:#8892b0">ℹ️ Landfill fraction is calculated automatically as 1 − (feed + biogas + compost)</p>'))
display(ip2)
display(out2)

---
## 📚 Summary & Discussion Questions

### Module 1 Key Takeaways
- A jump from 85% to 96% accuracy sounds small, but across 1,000 kg/day it **compounds into significant revenue and climate gains**.
- The payback curve shows how quickly capital investment in AI technology can be recovered.
- Each percentage point of accuracy improvement has a **quantifiable CO₂ benefit**.

### Module 2 Key Takeaways
- The **waste hierarchy** (feed > biogas > compost > landfill) maximises value recovery.
- Shifting from 80% to 5% landfill rate dramatically changes both the P&L and the carbon footprint.
- Methane from organic landfill waste is a potent GHG — **diversion has outsized climate value**.

### 💬 Discussion Questions
1. What real-world barriers prevent facilities from achieving high AI accuracy or high diversion rates?
2. How do **local market prices** for feed, biogas, and compost change the business case?
3. What happens to the payback period if AI hardware costs **drop by 50%**? Try it with the sliders!
4. At what **contamination level** does clean plastic become uneconomical to sell?
5. How would you use this model to pitch to a **municipal government** or an **impact investor**?

---
*Built for the Circular Economy Bootcamp · Python · Matplotlib · ipywidgets*